# LIME for Trading Model Explanation

This notebook demonstrates how to use LIME (Local Interpretable Model-agnostic Explanations) to explain predictions from trading models.

## What You'll Learn

1. How to fetch and prepare market data
2. How to train a trading model
3. How to generate LIME explanations for predictions
4. How to use LIME explanations to filter trading signals
5. How to backtest with and without LIME filtering

## Setup

First, let's install the required packages and import our modules.

In [ ]:
# Install required packages (uncomment if needed)
# !pip install yfinance scikit-learn pandas numpy matplotlib

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from data_loader import DataLoader, FeatureEngineering, prepare_training_data
from model import RandomForestTrader, XGBoostTrader, ModelEvaluator
from lime_explainer import LimeExplainer, LimeAnalyzer, create_lime_explainer_for_model
from backtest import Backtester, compare_with_without_lime, print_backtest_comparison

# Set random seed for reproducibility
np.random.seed(42)

# Configure matplotlib
plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

## 1. Load and Prepare Data

We'll fetch historical stock data and compute technical indicators.

In [ ]:
# Initialize data loader for Yahoo Finance
loader = DataLoader(source='yahoo')

# Fetch Apple stock data
symbol = 'AAPL'
df = loader.fetch_yahoo(
    symbol=symbol,
    start_date='2020-01-01',
    end_date='2024-01-01'
)

print(f"Loaded {len(df)} days of data for {symbol}")
df.tail()

In [ ]:
# Compute technical indicators
df_features = FeatureEngineering.compute_all_features(df)

print(f"\nFeatures computed: {len(df_features.columns)} columns")
print(f"Data points after feature engineering: {len(df_features)}")
print(f"\nFeature columns:")
print(df_features.columns.tolist())

In [ ]:
# Prepare features and target for training
X, y = prepare_training_data(
    df_features,
    target_column='return_1d',
    classification=True  # Predict UP (1) or DOWN (0)
)

print(f"Feature matrix shape: {X.shape}")
print(f"Target distribution:")
print(y.value_counts())

## 2. Train a Trading Model

We'll train a Random Forest classifier to predict price direction.

In [ ]:
# Split data into train and test sets
train_size = int(len(X) * 0.8)

X_train, X_test = X.iloc[:train_size], X.iloc[train_size:]
y_train, y_test = y.iloc[:train_size], y.iloc[train_size:]

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")

In [ ]:
# Train Random Forest model
model = RandomForestTrader(
    n_estimators=100,
    max_depth=10,
    min_samples_split=20,
    random_state=42
)

model.fit(X_train, y_train)
print("Model trained successfully!")

In [ ]:
# Evaluate the model
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)

metrics = ModelEvaluator.classification_metrics(y_test.values, y_pred)
metrics_proba = ModelEvaluator.classification_metrics_proba(y_test.values, y_proba)

print("Model Performance on Test Set:")
print("-" * 30)
for metric, value in {**metrics, **metrics_proba}.items():
    print(f"{metric}: {value:.4f}")

In [ ]:
# Display feature importance from the model
importance = model.get_feature_importance()

plt.figure(figsize=(10, 6))
importance.head(15).plot(kind='barh')
plt.xlabel('Feature Importance')
plt.title('Top 15 Most Important Features (Random Forest)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 3. LIME Explanations

Now let's create LIME explanations for individual predictions.

In [ ]:
# Create LIME explainer
explainer = create_lime_explainer_for_model(model, X_train)

print("LIME Explainer created!")
print(f"Number of features: {explainer.num_features}")
print(f"Number of samples for perturbation: {explainer.num_samples}")

In [ ]:
# Explain a single prediction
idx = 0  # First test sample
instance = X_test.iloc[idx]

explanation = explainer.explain(instance.values)

print(f"\nExplaining prediction for {X_test.index[idx].strftime('%Y-%m-%d')}")
print("=" * 50)
print(explanation)

In [ ]:
# Visualize the explanation
def plot_lime_explanation(explanation, top_n=10):
    """Plot LIME feature contributions."""
    df = explanation.to_dataframe().head(top_n)
    
    colors = ['green' if c > 0 else 'red' for c in df['contribution']]
    
    plt.figure(figsize=(10, 6))
    plt.barh(range(len(df)), df['contribution'], color=colors)
    plt.yticks(range(len(df)), df['feature'])
    plt.xlabel('Contribution to Prediction')
    plt.title(f"LIME Explanation\nPrediction: {'UP' if explanation.prediction == 1 else 'DOWN'} "
              f"(probability: {explanation.prediction_proba:.1%})")
    plt.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()

plot_lime_explanation(explanation)

## 4. Batch Explanations

Let's generate explanations for multiple predictions and analyze patterns.

In [ ]:
# Generate explanations for first 50 test samples
n_samples = 50
explanations = explainer.explain_batch(X_test.iloc[:n_samples].values)

print(f"Generated {len(explanations)} explanations")

In [ ]:
# Aggregate explanations to find overall important features
aggregated = LimeAnalyzer.aggregate_explanations(explanations)

print("Aggregated Feature Importance (LIME):")
print(aggregated.head(15).to_string())

In [ ]:
# Measure explanation consistency
consistency = LimeAnalyzer.explanation_consistency(explanations, top_n=5)
print(f"\nExplanation Consistency (top 5 features): {consistency:.2%}")
print("(Higher is better - indicates the model relies on similar features across predictions)")

In [ ]:
# Check local model quality distribution
r2_scores = [exp.local_model_score for exp in explanations]

plt.figure(figsize=(10, 4))
plt.hist(r2_scores, bins=20, edgecolor='black')
plt.xlabel('Local Model R² Score')
plt.ylabel('Frequency')
plt.title('Distribution of LIME Local Model Quality')
plt.axvline(x=0.5, color='red', linestyle='--', label='Threshold (0.5)')
plt.legend()
plt.tight_layout()
plt.show()

print(f"Mean R² score: {np.mean(r2_scores):.3f}")
print(f"Explanations with R² > 0.5: {sum(1 for s in r2_scores if s > 0.5)} / {len(r2_scores)}")

## 5. Backtesting with LIME Filtering

Let's compare trading performance with and without LIME-based signal filtering.

In [ ]:
# Prepare price data for backtesting
prices_test = df_features.loc[X_test.index][['open', 'high', 'low', 'close', 'volume']]

print(f"Backtest period: {prices_test.index[0].strftime('%Y-%m-%d')} to {prices_test.index[-1].strftime('%Y-%m-%d')}")
print(f"Number of trading days: {len(prices_test)}")

In [ ]:
# Compare backtest with and without LIME filtering
results_without, results_with = compare_with_without_lime(
    model=model,
    explainer=explainer,
    prices=prices_test,
    features=X_test,
    initial_capital=100000,
    transaction_cost=0.001,
    min_lime_score=0.5
)

# Print comparison
print_backtest_comparison(results_without, results_with)

In [ ]:
# Plot equity curves
plt.figure(figsize=(12, 6))

plt.plot(results_without.equity_curve, label='Without LIME Filtering', alpha=0.7)
plt.plot(results_with.equity_curve, label='With LIME Filtering', alpha=0.7)

plt.xlabel('Date')
plt.ylabel('Portfolio Value ($)')
plt.title('Backtest Equity Curves: LIME Filtering Comparison')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Analyzing Trade Explanations

Let's look at explanations for winning vs. losing trades.

In [ ]:
# Get winning and losing trades with explanations
winning_trades = [t for t in results_with.trades if t.return_pct > 0 and t.explanation is not None]
losing_trades = [t for t in results_with.trades if t.return_pct <= 0 and t.explanation is not None]

print(f"Winning trades with explanations: {len(winning_trades)}")
print(f"Losing trades with explanations: {len(losing_trades)}")

In [ ]:
# Compare average R² scores
if winning_trades and losing_trades:
    winning_r2 = np.mean([t.explanation.local_model_score for t in winning_trades])
    losing_r2 = np.mean([t.explanation.local_model_score for t in losing_trades])
    
    print(f"Average LIME R² for winning trades: {winning_r2:.3f}")
    print(f"Average LIME R² for losing trades: {losing_r2:.3f}")
    
    if winning_r2 > losing_r2:
        print("\nWinning trades have better LIME explanations on average!")
        print("This suggests that clearer explanations correlate with better trading outcomes.")

## Summary

In this notebook, we demonstrated:

1. **Data Preparation**: Loaded stock data and computed technical indicators
2. **Model Training**: Trained a Random Forest classifier for price prediction
3. **LIME Explanations**: Generated explanations showing which features drove predictions
4. **Explanation Analysis**: Aggregated explanations to understand model behavior
5. **LIME Filtering**: Used explanation quality to filter trading signals
6. **Trade Analysis**: Compared explanations for winning vs. losing trades

Key takeaways:
- LIME provides interpretable explanations for complex trading models
- Explanation quality (R² score) can be used to filter uncertain signals
- Understanding why a model makes predictions helps build trust and improve strategies